# Resonate — Scripture, where you already are

**A privacy-first context engine that weaves *verified* Scripture into the AI conversations people already have.**

This notebook is the *proof of work*: it runs the real matching engine end-to-end and shows the
evaluation metrics. The flagship surface is a Chrome extension that surfaces a quiet, dismissible
verse beside ChatGPT — but the engine here is surface-agnostic (it also powers a VS Code companion).

- **Gloo AI Studio** → reads emotion/themes, writes the one-line *bridge*, runs safety. *Never recites Scripture.*
- **YouVersion Platform API** → returns the verified, licensed verse text (anti-hallucination).
- **The engine** (this code) → hybrid retrieval (dense + BM25 + tags) fused with **RRF**, a temporal
  memory graph, a **Delivery Policy** (restraint), and a safety gate.

> Runs in **mock mode** here (no keys needed) so it's fully reproducible; the live Gloo + YouVersion
> integration flips on with one config change once challenge keys open (2026-07-06).
> **Enable *Internet* in the notebook settings** — the next cell does a one-time `git clone`.


In [ ]:
import os, sys
if not os.path.isdir('resonate'):
    !git clone -q https://github.com/krizz711/resonate
sys.path.insert(0, 'resonate')
import resonate
print('Resonate package at:', os.path.dirname(resonate.__file__))

## 1 · The engine in action

Give it real, messy human text; it finds the emotional *beats*, matches a verse, fetches the text,
and writes a bridge. Notice it **stays silent on non-resonant text** ('only speaks when it hears a story').


In [ ]:
from resonate import Engine, EngineConfig
engine = Engine(EngineConfig())

def show(text, user='nb'):
    r = engine.resonate(text, user)
    print('CONTEXT:', text)
    if not r['deliveries']:
        print('   -> (silent — no resonant beat)\n'); return
    for d in r['deliveries']:
        if d['status'] == 'delivered':
            print(f"   -> {d['reference']} ({d['translation']})  [themes={d['beat']['themes']}, fit={d['final']}]")
            print(f"      {d['verse_text']}")
            print(f"      bridge: {d['bridge']}")
            print(f"      considered: {[a['reference'] for a in d['alternatives']]}")
        elif d['status'] == 'safety_hold':
            print('   -> SAFETY HOLD (no verse):', d['message'][:90], '...')
    print()

show("I've been editing until 3am every night and I'm completely exhausted.")
show("Part of me wonders if any of this even matters.")
show("I'm so grateful — someone said my work helped them today.")

## 2 · Restraint &amp; safety — why it's *not* a pop-up

A **Delivery Policy** keeps it silent unless a real beat is present. And a crisis is caught on the raw
text (phrasing-robust) and routed to **a human-help card — never a verse**.


In [ ]:
show("what's the capital of France?")          # ordinary -> silence
show("honestly I do not want to live anymore")  # crisis    -> help, never a verse

## 3 · Memory over time — *being known, not searched*

The engine remembers the themes a person keeps returning to. After a theme recurs, it surfaces a
gentle note — personalization across conversations, not a single sentence.


In [ ]:
u = 'nb_memory'
for i in range(4):
    r = engine.resonate('I feel so anxious and worried about everything lately.', u)
    d = [x for x in r['deliveries'] if x['status'] == 'delivered']
    note = d[0].get('memory_note') if d else None
    print(f"message {i+1}:  verse={d[0]['reference'] if d else '-':<20}  memory_note={note}")

## 4 · Inside the match — hybrid retrieval + RRF

Three retrievers (dense embeddings, BM25, theme tags) each rank the verse shortlist; their *ranks*
are fused with **Reciprocal Rank Fusion**. Here are the top candidates for one beat, with each
retriever's rank (dense/sparse/tag):


In [ ]:
from resonate.providers.gloo import MockGloo
beat = MockGloo(EngineConfig()).segment("I'm stuck and I want to give up.")[0]
print('beat themes:', beat.themes, '| intensity:', round(beat.intensity, 2), '\n')
for c in engine.retriever.retrieve(beat, topk=6):
    r = c['ranks']
    print(f"{c['verse']['reference']:<22} RRF={c['rrf']:.4f}   ranks d/s/t = {r['dense']}/{r['sparse']}/{r['tag']}")

## 5 · Evaluation — proof it actually works

A 32-scenario labeled set measures theme recall, verse hit@1/@3, and — most importantly — **safety**.
These thresholds are also enforced by a unit test, so quality can't silently regress.


In [ ]:
!python resonate/eval/run_eval.py

## Architecture &amp; what's next

```
message -> segment (Gloo) -> beats -> hybrid retrieve (dense+BM25+tags, RRF)
        -> memory re-rank -> LLM verify (constrained) -> safety gate -> confidence
        -> fetch text (YouVersion) -> bridge (Gloo) -> deliver where you chose -> remember
```

Full design: [`ENGINE-DESIGN.md`](https://github.com/krizz711/resonate/blob/main/ENGINE-DESIGN.md) ·
Flagship extension: [`integrations/chatgpt-extension`](https://github.com/krizz711/resonate/tree/main/integrations/chatgpt-extension) ·
Repo: https://github.com/krizz711/resonate

*Mock mode here for reproducibility; the live Gloo + YouVersion integration is one config flag away.*
